# Compact boson partition function

This tutorial evaluates the compact boson partition function directly from a discretized ribbon graph and checks its radius dependence against the conventional period matrix expression.

In [1]:
from itertools import product

import numpy as np

from string_amplitudes import (
    compact_boson_partition_function,
    compute_period_map,
    generate_ribbon_graphs,
)

## Find the period matrix

We use the unique genus 1 ribbon graph with one face and equal edge lengths. Equal lengths gives the torus with $\tau=\tfrac12+i\tfrac{\sqrt3}{2}$.

In [2]:
# first we generate the associated ribbon graph structure. 
# We take the first entry of the list of ribbon graphs, since there is only one.
graph = generate_ribbon_graphs(genus=1, n_faces=1)[0]
edge_lengths = (40, 40, 40)
# compute_period_map returns an object containing the period matrix, the alpha-cycle
# normalized one-forms, and the genus.
period_data = compute_period_map(
    graph,
    edge_lengths,
    period_quadrature_order=128,
)
# the period matrix is 1-dimensional, so we can just take the [0,0] entry.
tau = period_data.period_matrix[0, 0]
expected_tau = 0.5 + 0.5j * np.sqrt(3.0)

print("Reconstructed tau:", tau)
print("Absolute error from the expected value:", abs(tau - expected_tau))
assert abs(tau - expected_tau) < 1e-10

Reconstructed tau: (0.49999999999999983+0.8660254037844386j)
Absolute error from the expected value: 1.6653345369377348e-16


## Comparing compact boson partition function to period matrix expression

Evaluate the analytic expression for the compact boson partition function in terms of the period matrix $\Omega$. The overall compact boson partition function is subject to Weyl anomaly, but the ratio of two partition functions at fixed moduli and varying radius is independent of Weyl anomaly:

$$
\frac{R_1\sum_{\mathbf m,\mathbf n\in\{-N,\ldots,N\}^{g}}\exp\!\left[-\pi R_1^{2}\,\operatorname{Re}\!\left((\mathbf m+\Omega\mathbf n)^{T}(\operatorname{Im}\Omega)^{-1}(\mathbf m+\overline{\Omega}\mathbf n)\right)\right]}
{R_2\sum_{\mathbf m,\mathbf n\in\{-N,\ldots,N\}^{g}}\exp\!\left[-\pi R_2^{2}\,\operatorname{Re}\!\left((\mathbf m+\Omega\mathbf n)^{T}(\operatorname{Im}\Omega)^{-1}(\mathbf m+\overline{\Omega}\mathbf n)\right)\right]}
$$

In [3]:
# evaluate the numerator or denominator of the above expression
# cutoff is the cutoff used in the sum.
def period_matrix_expression_compact_boson(radius, period_matrix, cutoff):
    period_matrix = np.asarray(period_matrix, dtype=np.complex128)
    genus = period_matrix.shape[0]
    y_inverse = np.linalg.inv(period_matrix.imag)
    integer_vectors = tuple(product(range(-cutoff, cutoff + 1), repeat=genus))

    total = 0.0
    for m_values in integer_vectors:
        m = np.asarray(m_values, dtype=float)
        for n_values in integer_vectors:
            n = np.asarray(n_values, dtype=float)
            vector = m + period_matrix @ n
            quadratic_form = float(
                np.real(vector @ y_inverse @ np.conjugate(vector))
            )
            total += np.exp(-np.pi * radius**2 * quadratic_form)
    return float(total)

## Compare radius ratios

Take the denominator to have $R=1$. The function `compact_boson_partition_function` computes the compact boson partition function in terms of the ribbon graph moduli.

## Convergence

As the number of discretization points used in the ribbon graph construction is increased, the accuracy of the computed period matrix should increase. Consequently, the accuracy of the compact boson partition function should increase.

In [5]:
# fix the numerator radius to be 1.5, and increase the number of
# discretization points in both the numerator and denominator.
test_radius = 1.5
print("edge length    relative difference at R=1.5")
for common_length in (12, 24, 40):
    lengths = (common_length,) * len(graph[0])
    omega = compute_period_map(
        graph, lengths, period_quadrature_order=128
    ).period_matrix

    ribbon_at_reference = compact_boson_partition_function(
        graph, lengths, reference_radius, lattice_cutoff=lattice_cutoff
    )
    ribbon_at_test = compact_boson_partition_function(
        graph, lengths, test_radius, lattice_cutoff=lattice_cutoff
    )
    ribbon_ratio = ribbon_at_test / ribbon_at_reference

    period_at_reference = period_matrix_expression_compact_boson(
        reference_radius, omega, lattice_cutoff
    )
    period_at_test = period_matrix_expression_compact_boson(
        test_radius, omega, lattice_cutoff
    )
    period_ratio = (
        test_radius * period_at_test
        / (reference_radius * period_at_reference)
    )
    difference = abs(ribbon_ratio - period_ratio) / abs(period_ratio)
    print(f"{common_length:11d}    {difference:10.3e}")

edge length    relative difference at R=1.5
         12     2.667e-03
         24     1.056e-03
         40     5.340e-04
